In [2]:
class ANDGate:
    def compute(self, a: int, b: int) -> int:
        return a & b

class ORGate:
    def compute(self, a: int, b: int) -> int:
        return a | b

class NOTGate:
    def compute(self, a: int) -> int:
        return 1 - a

class XORGate:
    def compute(self, a: int, b: int):
        return a ^ b

and_gate = ANDGate()
or_gate  = ORGate()
not_gate = NOTGate()
xor_gate = XORGate()

print("1 & 0 = ", and_gate.compute(1,0))
print("1 | 0 = ", or_gate.compute(1,0))
print("not 1 = ", not_gate.compute(1))
print("1 ^ 0 = ", xor_gate.compute(1,0))


1 & 0 =  0
1 | 0 =  1
not 1 =  0
1 ^ 0 =  1


In [3]:
class HalfAdder:
    def __init__(self) -> None:
        self.xorg : XORGate = XORGate()
        self.andg : ANDGate = ANDGate()

    def compute(self, a: int, b: int):
        sum_bit = self.xorg.compute(a, b)
        carry   = self.andg.compute(a, b)
        return sum_bit, carry

ha = HalfAdder()
print(f"Half Adder 1 + 1 = {ha.compute(1,1)}")

Half Adder 1 + 1 = (0, 1)


In [4]:
class FullAdder:
    def __init__(self) -> None:
        self.ha1 : HalfAdder = HalfAdder()
        self.ha2 : HalfAdder = HalfAdder()
        self.org : ORGate = ORGate()
    
    def compute(self, a: int, b: int, carry_in: int):
        s1, c1 = self.ha1.compute(a,b)
        s2, c2 = self.ha2.compute(s1, carry_in)
        carry_out = self.org.compute(c1, c2)

        return s2, carry_out

fa = FullAdder()
print(f"Full Adder 1 + 1 + 1 = {fa.compute(1,1,1)}")

Full Adder 1 + 1 + 1 = (1, 1)


In [5]:
class BinaryAdder:
    def __init__(self, bits:int=8) -> None:
        self.bits : int = bits
        self.full_adder : FullAdder = FullAdder()
    
    def add(self, a:int, b:int):
        carry  : int = 0
        result : int = 0

        for i in range(self.bits):
            a_bit = (a >> i) & 1
            b_bit = (b >> i) & 1

            s, carry = self.full_adder.compute(a_bit, b_bit, carry)
            result |= (s << i)
        
        return result

adder = BinaryAdder(8)
print(f"5 + 3 = {adder.add(5,3)}")

5 + 3 = 8


In [6]:
class Register:
    def __init__(self, name:str, size:int=8) -> None:
        self.name : str = name
        self.size : int = size
        self.value: int = 0
    
    def load(self, value:int) -> None:
        mask : int = (1 << self.size) - 1
        self.value : int = value & mask
    
    def read(self) -> int:
        return self.value
    
    def clear(self) -> None:
        self.value = 0

    def __repr__(self) -> str:
        return f"{self.name} : {self.value}"

ra:Register = Register("ra", 8)
ra.load(10)
print(ra)

ra : 10


In [7]:
class ProgramCounter(Register):
    def increment(self) -> None:
        self.value += 1
        

In [12]:
class ALU:
    def __init__(self, bits:int=8) -> None:
        self.bits     : int         = bits
        self.adder    : BinaryAdder = BinaryAdder(bits)
        self.and_gate : ANDGate     = ANDGate()
        self.or_gate  : ORGate      = ORGate()
        self.xor_gate : XORGate     = XORGate()

    def operate(self, op:str, a:int, b:int=0):
        if op == "ADD":
            return self.adder.add(a,b)
        elif op == "AND":
            return self.and_gate.compute(a,b)
        elif op == "OR":
            return self.or_gate.compute(a,b)
        elif op == "XOR":
            return self.xor_gate.compute(a,b)
        elif op == "PASS":
            return a
        else:
            raise ValueError("Unknown Operation")
        
alu = ALU()

print(f"ADD OPER: {alu.operate("ADD", 10, 20)}")
print(f"AND OPER: {alu.operate("AND", 10, 20)}")
print(f"XOR OPER: {alu.operate("XOR", 1, 1)}")
print(f"INV OPER: {alu.operate("INV", 1, 1)}")

ADD OPER: 30
AND OPER: 0
XOR OPER: 0


ValueError: Unknown Operation

In [ ]:
class Memory:
    def __init__(self, size:int=256) -> None:
        self.size  : int  = size
        self.cells : list = [0] * size
    
    def write(self, address:int, value:int):
        self.cells[address] = value
    
    def read(self, address:int) -> int:
        return self.cells[address]